# LINEAR EXPECTED SARSA

In [ ]:
from collections import defaultdict
from IPython import display
import gym
import matplotlib.pyplot as plt
import numpy as np

from src.utils.plots import plot_training_results

%matplotlib inline

In [ ]:
class LinearExpectedSarsa():

    def __init__(self, env, gamma=0.99, epsilon=1, alpha=0.01,init_factor=0.1):
        
        self.env = env
        na = env.action_space.n
        n_features = env.observation_space.shape[0]
        self.Qtheta = np.random.randn(na, n_features) * init_factor
        self.alpha = alpha
        self.epsilon = epsilon
        self.gamma = gamma
            
    def getQs(self, s):
        return np.dot(self.Qtheta, s) 

    def act(self, s):
        if np.random.rand() > self.epsilon:
            a = self.act_no_explore(s)
        else: 
            a = self.env.action_space.sample()
        return a

    def act_no_explore(self, s):
        return np.argmax(self.getQs(s))
        
    def learn(self, s, a, r, s1, done=False): 
        qsa = self.getQs(s)[a]
        if done:
            target = r
        else:
            max_value = np.max(self.getQs(s1))
            rnd_value = np.mean(self.getQs(s1))
            exp_value = (1 - self.epsilon) * max_value + self.epsilon * rnd_value
            target = r + self.gamma * exp_value
        self.Qtheta[a] = self.Qtheta[a] + self.alpha * (target - qsa) * s

        return (target - qsa) ** 2

In [ ]:
env = gym.make('LunarLander-v2')
env.reset()

nepisodes = 2000
n_log = 250

gamma = 0.99
epsilon = 1
alpha = 0.01
min_epsilon = 0.05
epsilon_decay = (epsilon - min_epsilon) / (nepisodes * 0.5)
min_alpha = 8e-7
alpha_decay = (alpha - min_alpha) / (nepisodes * 0.9)
history_rewards = []
history_losses = []

agent = LinearExpectedSarsa(env, gamma=gamma, epsilon=epsilon, alpha=alpha)

for episode in range(nepisodes):
    s = env.reset()
    cum_reward = 0
    episode_losses = []
    while True:
        a = agent.act(s)
        s1, r, done, _ = env.step(a)
        loss = agent.learn(s, a, r, s1, done)
        episode_losses.append(loss)
        cum_reward += r
        s = s1
        if done:
            agent.epsilon = max([agent.epsilon - epsilon_decay, min_epsilon])
            agent.alpha = max([agent.alpha - alpha_decay, min_alpha])
            mse = np.mean(episode_losses)
            history_rewards.append(cum_reward)
            history_losses.append(mse)
            if not episode % n_log:
                print(
                    f'Episode: {episode}, '
                    f'rew: {np.round(np.mean(history_rewards[-n_log:]), 2)}, '
                    f'mse: {np.round(np.mean(history_losses[-n_log:]), 2)}, '
                    f'eps: {np.round(agent.epsilon, 2)}, '
                    f'alpha: {np.round(agent.alpha, 5)}'
                )
            break    

In [ ]:
plot_training_results(
    rewards=history_rewards,
    losses=history_losses,
    window=100,
    scale="linear"
)

In [ ]:
env = gym.make('LunarLander-v2')
s = env.reset()
img = plt.imshow(env.render(mode='rgb_array'))
cum_reward = 0
done = False
step = 0
while True:
    img.set_data(env.render(mode='rgb_array'))
    plt.title(f'Step: {step}, rew: {np.round(cum_reward)}')
    plt.axis('off')
    display.display(plt.gcf())
    display.clear_output(wait=True)
    if done:
        break
    action = agent.act_no_explore(s)
    s, r, done, _ = env.step(action)
    step += 1
    cum_reward += agent.gamma ** step * r

env.close()
print(f"Done Rew: {np.round(agent.gamma ** step * r, 2)}")

In [ ]:
agent.Qtheta